# DSE 200 — Day 2 Lab
## Python Mastery, Matplotlib, Pandas, Regression

**Chunks 4.1, 4.2, 4.3 and 6 · agent off for hands-on**

Section numbers match the slides.

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while Rick talks. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost — every section rebuilds its own data.

In [ ]:
# ▶ SETUP — run first. Finds the data and defines check_read().
import os, textwrap
import pandas as pd, numpy as np

CANDIDATES = ["movielens", "/content/movielens", "data/movielens", "../movielens", "."]
ML = next((d for d in CANDIDATES if os.path.exists(os.path.join(d, "movies.csv"))), None)
if ML is None:  # opened from GitHub: Colab brought the notebook but not the data
    try:
        import urllib.request
        for name in ("movies.csv", "ratings_reduced.csv"):
            urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
        ML = "."
        print("downloaded movies.csv and ratings_reduced.csv from the course repo")
    except Exception as e:
        print("could not download the MovieLens files:", e)
assert ML, ("movies.csv not found -- drag movies.csv and ratings_reduced.csv into the Files pane "
            "(folder icon, left), or run: from google.colab import files; files.upload()")

movies  = pd.read_csv(os.path.join(ML, "movies.csv"))
RATINGS = ("ratings_reduced.csv" if os.path.exists(os.path.join(ML, "ratings_reduced.csv"))
           else "ratings.csv")
ratings = pd.read_csv(os.path.join(ML, RATINGS))

print(f"movies  {movies.shape}   from {ML}/movies.csv")
print(f"ratings {ratings.shape}  from {ML}/{RATINGS}")

_READ = {
"4.1": ("['Toy Story', 'Jumanji', 'Heat']",
        "Split each title at ' (' and keep the first piece -- but only for titles\n"
        "that actually end in a bracket. The `if` filters, the expression transforms.\n"
        "Note it is on three lines. A comprehension does not have to be on one."),
"4.2": ("3.5\nNone",
        "add_rating PRINTS but never RETURNS, so the call hands back None.\n"
        "This is the number one source of 'NoneType has no attribute...' two cells later.\n"
        "If a function computes something, it must return it."),
"4.5": ("mean rating per movie, for movies with enough votes, above a threshold",
        "Renaming d->ratings_by_movie, x->threshold, y->min_votes, r->means makes it\n"
        "readable with zero behaviour change. That is the entire lesson."),
"4.12":("Yes -- both give the same number.",
        "query() takes a string, so no brackets and no &. Nicer for long conditions,\n"
        "slower on huge frames, and it cannot see your Python variables unless you\n"
        "prefix them with @ -- e.g. ratings.query('rating > @cutoff')."),
"4.14":("Both. Smaller with an inner join, bigger with duplicate keys on the right.",
        "An inner join silently DROPS rows that matched nothing.\n"
        "A right side with repeated keys MULTIPLIES rows.\n"
        "Neither raises an error. Print len() before and after, every time."),
"4.19":("No. R2 is 0.055 -- the line explains 5.5% of why films differ.",
        "The slope is real and measurable. The line is nearly useless for predicting\n"
        "any particular film. Always report R2 in the same breath as a slope."),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print("check_read() ready.")

# Python Mastery

*Slides D02.02 · sections 4.1–4.6 · comprehensions, functions, errors, readable code · exercise 4.6*

## 4.1 Comprehensions

A loop that builds a list, on one line. Read it left to right: **what you want**, then **where it comes from**.

```
[  expression   for x in things   if condition  ]
```

In [ ]:
# ▶ DEMO
titles = ["Toy Story (1995)", "Jumanji (1995)", "Heat (1995)"]

lengths = []                       # the loop
for t in titles:
    lengths.append(len(t))

lengths2 = [len(t) for t in titles]     # the same thing
print(lengths, lengths2, lengths == lengths2)

print([t for t in titles if "1995" in t])       # with a filter
print([t.upper() for t in titles if t.startswith("T")])

In [ ]:
# ▶ DEMO — dicts and sets too
genres = ["Adventure|Animation|Children|Comedy|Fantasy",
          "Adventure|Children|Fantasy",
          "Comedy|Romance"]

print({t: len(t) for t in titles})                       # dict comprehension
print({g for row in genres for g in row.split("|")})     # set -- dedupes as it builds
print([(x, y) for x in range(2) for y in range(2)])      # two for clauses

In [ ]:
# 🔍 READ 4.1 — what does this build? Say it in one sentence.
#
#     result = [t.split(" (")[0]
#               for t in titles
#               if t.endswith(")")]
#
# Commit first.

prediction = "???"
check_read("4.1", prediction)

**Where to stop.** One `for` — always clearer. One `for` + one `if` — still clearer. Two `for` clauses — borderline, break the lines. Three, or nested comprehensions — **write the loop**.

If you cannot read it aloud in one breath, it is showing off. An agent will happily hand you a four-clause comprehension; you are allowed to rewrite it.

## 4.2 Functions

Write it once, name it well.

In [ ]:
# ▶ DEMO
def parse_year(title):
    """Pull the release year out of a MovieLens title."""
    return int(title.split("(")[-1].rstrip(")"))

print(parse_year("Toy Story (1995)"))
print(parse_year.__doc__)

In [ ]:
# 🔍 READ 4.2 — two lines print. What are they?
#
#     def add_rating(r):
#         print(r + 0.5)
#
#     result = add_rating(3.0)
#     print(result)
#
# Commit first.

prediction = "???"
check_read("4.2", prediction)

In [ ]:
# ▶ DEMO — defaults, keyword args, and forcing the names
def top_rated(df, *, n=5, min_votes=50, ascending=False):
    """Everything after * MUST be passed by name."""
    return f"n={n} min_votes={min_votes} ascending={ascending}"

print(top_rated(ratings, n=10, min_votes=100))
try:
    top_rated(ratings, 10, 100, False)
except TypeError as e:
    print("TypeError:", e)

In [ ]:
# ▶ DEMO — the mutable-default trap
def collect_bad(x, bucket=[]):
    bucket.append(x); return bucket

print(collect_bad(1), collect_bad(2), "<- ???")

def collect_good(x, bucket=None):
    if bucket is None: bucket = []
    bucket.append(x); return bucket

print(collect_good(1), collect_good(2))

In [ ]:
# ▶ DEMO — scope
MIN_VOTES = 50

def is_popular(n):
    threshold = MIN_VOTES        # reading an outside name is fine
    return n >= threshold

print(is_popular(80))
try:
    threshold
except NameError as e:
    print("NameError:", e, "<- it died with the function")

## 4.3 Functions Are Values

You can store one in a variable, put it in a dict, and pass it to something else.

In [ ]:
# ▶ DEMO
f = parse_year                      # no parentheses -- the function itself
print(f("Heat (1995)"))

ops = {"mean": np.mean, "median": np.median, "max": np.max}
for name, fn in ops.items():
    print(f"{name:>7}: {fn(ratings.rating):.3f}")

In [ ]:
# ▶ DEMO — lambda, which you have already used
films = [{"title": "A", "rating": 4.4}, {"title": "B", "rating": 3.1},
         {"title": "C", "rating": 4.9}]

print(sorted(films, key=lambda m: m["rating"], reverse=True))
print(max(films, key=lambda m: m["rating"])["title"])

## 4.4 Errors On Purpose

Not every MovieLens title has a year — and your parser fails on more of them than you think.

In [ ]:
# ▶ DEMO — how badly does the naive parser do?
def parse_year_naive(title):
    return int(title.split("(")[-1].rstrip(")"))

broken = []
for t in movies.title:
    try:
        parse_year_naive(t)
    except ValueError:
        broken.append(t)

print(f"{len(broken)} of {len(movies)} titles break it ({len(broken)/len(movies):.2%})")
for t in broken[:6]:
    print("   ", t)

Only **19** of those have no year at all. **206** have one, followed by a trailing space after the `)`. The last 3 are a date range or a still-running series: `(2007-)`, `(2009– )`, `(1975-1979)`.

**Your function fails twelve times more often than the data is broken.** Know which is which before you blame the file.

In [ ]:
# ▶ DEMO — survive it
def parse_year(title):
    """Release year as an int, or None if there is not one we can read."""
    try:
        return int(title.split("(")[-1].rstrip(")"))
    except (ValueError, AttributeError):
        return None

print(parse_year("Toy Story (1995)"), parse_year("Babylon 5"))

# catch the SPECIFIC error. A bare `except:` swallows your own typos too.

In [ ]:
# ▶ DEMO — raise, when the caller made the mistake
def mean_rating(values):
    if len(values) == 0:
        raise ValueError("mean_rating() needs at least one rating")
    return sum(values) / len(values)

try:
    mean_rating([])
except ValueError as e:
    print("ValueError:", e)

# Compare that to "ZeroDivisionError: division by zero".
# One of them tells you how to fix it.

## 4.5 Code You Can Read Next Week

> *"Pythonic code is clear and easy to read."* — readability counts.

Type `import this` in a cell sometime. It is a joke, and it is also the design philosophy.

In [ ]:
# 🔍 READ 4.5 — what does this do? You have sixty seconds.
#
#     def f(d, x, y=2):
#         r = {}
#         for k, v in d.items():
#             if len(v) >= y:
#                 r[k] = sum(v) / len(v)
#         return {k: v for k, v in r.items() if v > x}
#
# Say it in one sentence. Commit first.

prediction = "???"
check_read("4.5", prediction)

In [ ]:
# ▶ DEMO — identical behaviour, named properly
def mean_above(ratings_by_movie, threshold, min_votes=2):
    """Mean rating per movie, keeping only well-voted films above a threshold."""
    means = {}
    for movie, votes in ratings_by_movie.items():
        if len(votes) >= min_votes:
            means[movie] = sum(votes) / len(votes)
    return {m: avg for m, avg in means.items() if avg > threshold}

demo = {"Shawshank": [5,5,4,5], "Obscure": [5], "Meh": [2,3,2,3]}
print(mean_above(demo, threshold=4.0, min_votes=2))

| Instead of | Write |
| --- | --- |
| `d`, `r`, `x`, `tmp` | what it actually holds |
| a function called `process` | a **verb**: `parse_year`, `mean_above` |
| three levels of nested `if` | early `return`s |
| `if len(xs) > 0:` | `if xs:` |
| `if x == None:` | `if x is None:` |

An agent will write code that works and is unreadable. It has no stake in whether you can maintain it. **You are the one who has to explain it.**

## 4.6 Your Turn — 15 minutes

In [ ]:
# ✍️ YOU TRY 4.6
# 1. parse_year(title)  -- int, or None. Must not crash on "Babylon 5". Docstring required.
# 2. clean_titles       -- ONE comprehension: every title with its year stripped,
#                          only for titles that have a parsable year.
# 3. genre_counts(rows) -- dict of genre -> how many films carry it.
#                          Genres arrive pipe-separated: "Adventure|Children".

def parse_year(title):
    pass  # TODO

clean_titles = []   # TODO

def genre_counts(rows):
    return {}  # TODO -- a dict of genre -> count

print(parse_year("Heat (1995)"), parse_year("Babylon 5"))
print(clean_titles[:3], len(clean_titles))
gc = genre_counts(movies.genres)
print(sorted(gc.items(), key=lambda p: -p[1])[:3])

In [ ]:
# ✅ CHECKPOINT 4.6
assert parse_year("Heat (1995)") == 1995
assert parse_year("Babylon 5") is None, "must return None, not raise"
assert parse_year.__doc__, "give it a docstring"

assert len(clean_titles) == 27050, f"expected 27050 titles, got {len(clean_titles)}"
assert clean_titles[0] == "Toy Story", clean_titles[0]
assert all("(" not in t for t in clean_titles[:200]), "strip the year off"

gc = genre_counts(movies.genres)
assert gc["Drama"] == 13344, gc.get("Drama")
assert gc["Comedy"] == 8374, gc.get("Comedy")
assert gc["Thriller"] == 4178, gc.get("Thriller")
assert gc["(no genres listed)"] == 246, "246 films have no genre. What would you do with them?"
print("Correct.")

# Matplotlib and Data Storytelling

*Slides D02.03 · sections 4.7–4.10 · figure & axes, four charts, storytelling · exercise 4.10*

## 4.7 Anatomy of a Figure

- **Figure** — the whole canvas
- **Axes** — one plot inside it (confusingly, "axes" means the *plot*)

Learn one style: `fig, ax = plt.subplots()`. The `plt.plot(...)` style relies on a hidden "current figure" and falls apart the moment you want two panels.

In [ ]:
# ▶ DEMO
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ratings.rating, bins=20, edgecolor="white")
ax.set_xlabel("rating (stars)")
ax.set_ylabel("number of ratings")
ax.set_title("4 stars is the most common rating, by a margin")
plt.show()

## 4.8 Four Charts

| Question | Chart |
| --- | --- |
| How is this **one** thing spread out? | histogram |
| Do these **two** things move together? | scatter |
| How do these **categories** compare? | bar |
| How did this change **over time**? | line |

Start with the question, not the chart.

In [ ]:
# ▶ DEMO — histogram. Look at the shape before you style it.
whole = ratings.rating[ratings.rating == ratings.rating.round()].size / len(ratings)
print(f"whole stars: {whole:.1%}   half stars: {1-whole:.1%}")
print(ratings.rating.value_counts(normalize=True).sort_index().round(3).to_dict())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ratings.rating, bins=20, edgecolor="white")
ax.axvline(ratings.rating.mean(), color="crimson",
           label=f"mean {ratings.rating.mean():.2f}")
ax.set_xlabel("rating (stars)"); ax.set_ylabel("number of ratings")
ax.set_title("Three quarters of ratings are whole stars -- people round")
ax.legend(); plt.show()

In [ ]:
# ▶ DEMO — scatter. alpha is not decoration.
stats = ratings.groupby("movieId").agg(n=("rating", "count"), avg=("rating", "mean"))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(stats.n, stats.avg, s=6, alpha=0.15)
ax.set_xscale("log")
ax.set_xlabel("number of ratings (log scale)"); ax.set_ylabel("mean rating")
ax.set_title("Films with few votes scatter far wider")
plt.show()

few  = stats[stats.n <= 2].avg.std()
many = stats[stats.n >= 100].avg.std()
print(f"sd with <=2 votes: {few:.2f}   with >=100: {many:.2f}   -> {few/many:.1f}x wider")

In [ ]:
# ▶ DEMO — bar. SORT IT.
counts = movies.genres.str.split("|").explode().value_counts().head(10)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(counts.index[::-1], counts.values[::-1])
ax.set_xlabel("number of films"); ax.set_xlim(0, None)
ax.set_title("Drama is in half of all 27,278 films")
plt.show()

In [ ]:
# ▶ DEMO — pandas draws, you finish. Hand pandas your ax, then finish with the same ax. calls.
genre_counts = movies.genres.str.split("|").explode().value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
genre_counts.head(10).sort_values().plot.barh(ax=ax)   # ascending -> the biggest bar lands on TOP
ax.set_xlabel("number of films"); ax.set_ylabel("")
ax.set_title("Drama is in half of all 27,278 films")
plt.show()

In [ ]:
# ▶ DEMO — get it out. Save BEFORE plt.show(): plt.savefig after show() writes a blank image.
fig, ax = plt.subplots(figsize=(7, 4))
genre_counts.head(10).sort_values().plot.barh(ax=ax)
ax.set_xlabel("number of films"); ax.set_ylabel("")
ax.set_title("Drama is in half of all 27,278 films")
fig.savefig("genres.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved genres.png --", os.path.getsize("genres.png"), "bytes. Find it in the Files pane (folder icon, left).")

In [ ]:
# ▶ DEMO — line. Only when x is ordered.
years = movies.title.str.extract(r"\((\d{4})\)$")[0].dropna().astype(int)
per_year = years[years.between(1920, 2015)].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(per_year.index, per_year.values)
ax.set_xlabel("release year"); ax.set_ylabel("films in the catalogue")
ax.set_ylim(0, None)
ax.set_title("The catalogue is dominated by recent films")
plt.show()

## 4.9 Storytelling

**Three rules.**

1. **Sorted, not alphabetical.** Alphabetical is the order the computer found them in. Sorted is a finding.
2. **Start at zero, label with units.** Crime has 2,939 films and Action 3,520 — a 20% gap. Start the axis at 2,900 and Action looks *sixteen times* bigger.
3. **The title is the finding, not the contents.**

| Weak | Better |
| --- | --- |
| "Genre Chart" | "Drama is in half of all 27,278 films" |
| "Rating distribution" | "Three quarters of ratings are whole stars" |

In [ ]:
# ▶ DEMO — the same data, honestly and dishonestly
sub = movies.genres.str.split("|").explode().value_counts()[["Action","Crime"]]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(sub.index, sub.values); axes[0].set_ylim(0, None)
axes[0].set_title("starts at zero"); axes[0].set_ylabel("films")
axes[1].bar(sub.index, sub.values); axes[1].set_ylim(2900, None)
axes[1].set_title("starts at 2,900"); axes[1].set_ylabel("films")
fig.tight_layout(); plt.show()

print(sub.to_dict(), "-- a 20% gap, drawn two ways")

In [ ]:
# ▶ DEMO — point at the finding. The title says WHAT; the arrow shows WHERE.
four = (ratings.rating == 4.0).sum()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ratings.rating, bins=20, edgecolor="white")
ax.set_xlabel("rating (stars)"); ax.set_ylabel("number of ratings")
ax.set_title("4 stars is the most common rating, by a margin")
ax.annotate(f"4 stars: {four / len(ratings):.0%} of every rating",
            xy=(4, four),            # the point you mean
            xytext=(1.0, 120000),    # where the words go
            arrowprops=dict(arrowstyle="->"))
plt.show()

**The question you will be asked:** *"So what should we do about it?"*

Before you show a chart, write the sentence you would say out loud while it is on screen. If you cannot write the sentence, the chart is not finished — or it is the wrong chart. That sentence is also your title.

## 4.10 Your Turn — 8 minutes

In [ ]:
# ✍️ YOU TRY 4.10
# 1. A histogram of all ratings, mean marked, both axes labelled with units.
# 2. A SORTED horizontal bar of the ten most common genres, starting at zero.
# 3. Give ONE of them a title that is a finding -- a full sentence.

# TODO

**Write your sentence here** (double-click to edit):

> What would you tell someone to *do* about what this chart shows?

*(your answer)*

The sentence is the graded part. Anyone can call `ax.hist`.

# Pandas Deep Dive

*Slides D02.04 · sections 4.11–4.16 · selecting, groupby, merge, time · activity 4.16*

## 4.11 The Mental Model

A DataFrame is a **dictionary of columns**. Each column is a Series; all columns share one Index.

Last week you built `{"Seattle": [8.4, 11.2], "Austin": [19.0]}` by hand. This is that, grown up.

In [ ]:
# ▶ DEMO — the four calls you run on every new file, forever
print(ratings.shape)
print(ratings.dtypes)
print(ratings.head(3))
print(ratings.describe())

In [ ]:
# ▶ DEMO — Series, Index, and what is missing
print(type(ratings["rating"]).__name__, "| index:", ratings.index)
print("\nmovies columns :", list(movies.columns))
print("ratings columns:", list(ratings.columns))
print("\nratings has no titles. movies has no ratings.")
print("You cannot answer 'what is the best film?' from either alone.")

## 4.12 Selecting and Filtering

`iloc` is positional, like a list. `loc` is by label, like a dictionary. And masks are exactly what you did in NumPy last week.

In [ ]:
# ▶ DEMO
print(ratings["rating"].head(3))            # one column -> Series
print(ratings[["userId","rating"]].head(3)) # several -> DataFrame
print(ratings.iloc[0])                      # by POSITION
print(ratings.loc[0, "rating"])             # by LABEL

In [ ]:
# ▶ DEMO — masks
print((ratings["rating"] > 4).head(3).tolist())
print(len(ratings[ratings["rating"] > 4]), "ratings above 4")

# brackets round EVERY comparison, same rule as NumPy
sub = ratings[(ratings["rating"] >= 4) & (ratings["userId"] < 100)]
print(len(sub))

In [ ]:
# 🔍 READ 4.12 — will these give the same answer?
#
#   a = ratings[ratings["rating"] > 4]["userId"].nunique()
#   b = ratings.query("rating > 4")["userId"].nunique()
#
# Commit first.

prediction = "???"
check_read("4.12", prediction)

## 4.13 groupby — the one you built by hand

**Split, apply, combine.** Last week it was twelve lines and a `setdefault`. Today it is one.

In [ ]:
# ▶ DEMO
print(ratings.groupby("movieId")["rating"].mean().head(3))

# name your output columns as you build them
stats = ratings.groupby("movieId").agg(n=("rating", "count"),
                                       avg=("rating", "mean"))
print(stats.head(3))
print("films rated:", len(stats))

In [ ]:
# ▶ DEMO — the filter that matters
print("best films, no vote filter:")
print(stats.nlargest(5, "avg"))
print(f"\n{(stats.avg == 5.0).sum()} films have a perfect 5.0 -- mostly on one vote.")
print("\nbest films with at least 50 votes:")
print(stats[stats.n >= 50].nlargest(3, "avg"))

**Always ask how many rows are behind each group.** That is the difference between a chart and a finding.

## 4.14 Merge

`stats` has movieIds. Nobody knows what movieId 318 is.

In [ ]:
# ▶ DEMO
best = stats.merge(movies, on="movieId", how="left")
print(best[best.n >= 100].nlargest(3, "avg")[["title","avg","n"]].to_string(index=False))

In [ ]:
# 🔍 READ 4.14 — the check everyone skips
#
#   before = len(stats)
#   merged = stats.merge(movies, on="movieId", how="left")
#   after  = len(merged)
#
# Can `after` be BIGGER than before? SMALLER? Both?  Commit first.

prediction = "???"
check_read("4.14", prediction)

In [ ]:
# ▶ DEMO — prove both directions on a tiny frame
left  = pd.DataFrame({"k":[1,2,3], "v":["a","b","c"]})
dupes = pd.DataFrame({"k":[1,1,2], "w":["x","y","z"]})   # k=1 appears TWICE
missing = pd.DataFrame({"k":[1], "w":["x"]})             # k=2,3 absent

print("left rows:", len(left))
print("inner join with missing keys ->", len(left.merge(missing, on="k", how="inner")), "SMALLER")
print("left join with duplicate keys ->", len(left.merge(dupes, on="k", how="left")), "BIGGER")
print("\nNeither raised an error.")

In [ ]:
# ▶ DEMO — so: three lines, every time
before = len(stats)
merged = stats.merge(movies, on="movieId", how="left")
print(before, "->", len(merged))
print("unmatched titles:", merged.title.isna().sum())
print("is the key unique on the right?", not movies.movieId.duplicated().any())

## 4.15 Time

That integer column is a calendar in disguise — seconds since 1 January 1970.

In [ ]:
# ▶ DEMO
print(ratings.timestamp.head(3).tolist())

r = ratings.copy()
r["ts"] = pd.to_datetime(r.timestamp, unit="s")
print(r.ts.min().date(), "..", r.ts.max().date())

print(r.ts.dt.year.head(3).tolist(), r.ts.dt.dayofweek.head(3).tolist())

In [ ]:
# ▶ DEMO — groupby on a piece of the date
by_year = r.groupby(r.ts.dt.year)["rating"].agg(["count","mean"]).round(3)
print(by_year.loc[[1996, 2000, 2013]])

print("\n1996 mean:", round(by_year.loc[1996,'mean'],3),
      " 2013 mean:", round(by_year.loc[2013,'mean'],3))
print("Are people getting more generous -- or is something else going on?")
print("Hold that. Section 4.17.")

## 4.16 In-Class Activity — 15 minutes

In [ ]:
# ✍️ YOU TRY 4.16
# 1. n_users, n_films  -- distinct users and films in the ratings file
# 2. stats             -- groupby + agg, columns named `n` and `avg`
# 3. merged            -- merge in the titles. Record row count before and after.
# 4. top10             -- the ten best films with AT LEAST 100 votes (titles + avg + n)
# 5. dow               -- mean rating by day of week. Is there a weekend effect?

n_users = n_films = 0     # TODO
stats   = None            # TODO
merged  = None            # TODO
rows_before = rows_after = 0   # TODO
top10   = None            # TODO
dow     = None            # TODO

In [ ]:
# ✅ CHECKPOINT 4.16
assert n_users == 3397, n_users
assert n_films == 12280, n_films
assert list(stats.columns) == ["n","avg"], list(stats.columns)
assert len(stats) == 12280
assert rows_before == rows_after == 12280, (rows_before, rows_after)
assert merged.title.isna().sum() == 0, "every movieId should match"
assert top10.iloc[0].title.startswith("Shawshank"), top10.iloc[0].title
assert abs(top10.iloc[0].avg - 4.4689) < 0.001, top10.iloc[0].avg
assert abs(dow.max() - dow.min() - 0.1324) < 0.01, dow.max()-dow.min()
print("Correct.")

# Linear Regression

*Slides D02.05 · sections 4.17–4.20 · fit a line, read R², argue with it · exercise 4.20*

## 4.17 Linear Regression — the idea

```
y = m·x + b
```

**Fitting** means finding the `m` and `b` that make the line miss the actual points by as little as possible — measured by adding up the squared vertical gaps. That is all *least squares* means.

We left a question open at 4.15. Two different questions hide in it:

1. Do **raters** get more generous over time?
2. Do **older films** score higher than newer ones?

In [ ]:
# ▶ SETUP 4.17 — from cold. Films with a readable year and at least 50 votes.
stats = ratings.groupby("movieId").agg(n=("rating","count"), avg=("rating","mean"))
d = stats.merge(movies, on="movieId", how="left")
d["year"] = d.title.str.extract(r"\((\d{4})\)$")[0].astype("float")
d = d[(d.n >= 50) & d.year.notna()].copy()
print(len(d), "films with >=50 votes and a readable year")
print(d[["title","year","n","avg"]].head(3).to_string(index=False))

## 4.18 Doing It

In [ ]:
# ▶ DEMO — NumPy, one line
m, b = np.polyfit(d.year, d.avg, 1)
print(f"slope {m:.5f} stars/year   intercept {b:.2f}")
print(f"over 60 years that is {m*60:+.2f} stars")
print(f"1950 -> {m*1950+b:.2f}    2010 -> {m*2010+b:.2f}")

In [ ]:
# ▶ DEMO — scikit-learn, the shape you keep
from sklearn.linear_model import LinearRegression

X = d[["year"]]        # 2-D: one column per feature
y = d["avg"]

model = LinearRegression().fit(X, y)
print("slope    ", round(model.coef_[0], 5))
print("intercept", round(model.intercept_, 2))
print("R2       ", round(model.score(X, y), 4))

# Every model on Days 3 and 4 has this exact shape.

## 4.19 Reading the Output

In [ ]:
# 🔍 READ 4.19
#
#   slope     = -0.0075 stars per year
#   1950 film -> predicted 3.78
#   2010 film -> predicted 3.33
#   R2        =  0.055
#
# Is "older films are better" a finding? Commit to yes or no.

prediction = "???"
check_read("4.19", prediction)

**R² is the share of variation your line explains**, from 0 to 1.

`R² = 0.055` means the line explains **5.5%** of why films differ in rating. The other 94.5% is genre, budget, director, luck.

| R² | Reading |
| --- | --- |
| 0.9 | the line is most of the story |
| 0.5 | a real, partial pattern |
| 0.05 | a slope you can measure and should not lean on |

**Always report R² next to a slope. A slope alone is half a sentence.**

In [ ]:
# ▶ DEMO — and the confound
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(d.year, d.avg, s=8, alpha=0.2)
xs = np.array([d.year.min(), d.year.max()])
ax.plot(xs, m*xs + b, color="crimson", linewidth=2)
ax.set_xlabel("release year"); ax.set_ylabel("mean rating")
ax.set_title(f"slope {m:.4f}/yr, R2 {model.score(X,y):.3f} -- the line explains 5.5%")
plt.show()

print("Which OLD films are in MovieLens? The ones people still watch.")
print("Forgotten films from 1955 were never rated. That is SURVIVORSHIP BIAS.")
print("We did not measure 'films get worse'. We measured 'bad old films disappear'.")

## 4.20 Your Turn — 8 minutes

Fit one, then argue with it.

In [ ]:
# ✍️ YOU TRY 4.20
# 1. slope, r2       -- fit year -> avg on `d` (>=50 votes). Report both.
#    np.polyfit gives slope and intercept, not R2. For one feature: r2 = np.corrcoef(x, y)[0, 1] ** 2
# 2. slope_pre, slope_post -- fit pre-1990 and post-1990 SEPARATELY.
# 3. Do the slopes agree?

slope = r2 = 0.0            # TODO
slope_pre = slope_post = 0.0  # TODO

print(f"all      slope {slope:+.5f}  R2 {r2:.4f}")
print(f"pre-1990 slope {slope_pre:+.5f}")
print(f"post-1990 slope {slope_post:+.5f}")

In [ ]:
# ✅ CHECKPOINT 4.20
assert abs(slope - (-0.00749)) < 0.0005, slope
assert abs(r2 - 0.0554) < 0.005, r2
assert slope_pre < 0 < slope_post, (slope_pre, slope_post)
print("Correct.\n")
print("Look at what you just found: the slope FLIPS SIGN.")
print(f"  pre-1990  {slope_pre:+.4f}   before 1990, OLDER films rate higher")
print(f"  post-1990 {slope_post:+.4f}  from 1990 on, NEWER films rate higher")
print("\nThe overall slope was an average of two opposite trends.")
print("That is the third question from the slides: does it survive a split?")
print("Here, emphatically, it does not. You do not have a finding.")

**Write your sentence here** (double-click to edit):

> One sentence you would actually be willing to say out loud about films and age.

*(your answer)*

---

**Model answer**

*"In this catalogue, films released before 1990 and after 1990 show opposite rating trends, the overall relationship explains about 5% of the variation, and the sample is biased toward old films people still watch — so I would not claim anything about film quality and age from this data."*

#### Three questions, every time

1. What is the **R²**? Say it in the same breath as the slope.
2. What is **missing** from this data? Survivorship, selection, dropped rows.
3. Does it **survive a split**? If the slope flips, you do not have a finding.

## Wrap

You can now:

- Replace a build-a-list loop with a comprehension — and know when not to
- Write a function with a docstring, sensible defaults, and named arguments
- Catch the error you expect, raise one that says what to do
- Build a figure with `fig, ax`, choose the chart from the question, and title it with a finding
- `groupby` + `agg` with named columns, and filter on group size
- `merge` two frames and **check the row count every time**
- Fit a line, read its R², and argue with it

**Tomorrow:** EDA and classification on customer data — where a good metric can hide a useless model.